In [1]:
import time
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier

In [2]:
X_train_df = pd.read_csv("X_train_features.csv")
y_train = np.load("y_train.npy")
feature_names = joblib.load("feature_names.joblib")

print("Training Shape :", X_train_df.shape)
print(pd.Series(y_train).value_counts())

Training Shape : (167766, 60)
0    83883
1    83883
Name: count, dtype: int64


In [3]:
lgbm_param_dist = {
    "n_estimators": [100, 200, 400],
    "num_leaves": [15, 31, 63],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [-1, 5, 10],
    "subsample": [0.7, 0.9, 1.0],
}

lgbm_search = RandomizedSearchCV(
    LGBMClassifier(random_state=42,verbosity=-1),
    param_distributions=lgbm_param_dist,
    n_iter=15, cv=3, scoring="f1", random_state=42, n_jobs=-1
)

In [4]:
# ============================================================
# TRAIN MODEL
# ============================================================
print("="*60)
print("Training LightGBM...")
print("="*60)

start = time.time()
lgbm_search.fit(X_train_df, y_train)
end = time.time()

lgbm = lgbm_search.best_estimator_
lgbm_training_time = end - start

print("Best params:", lgbm_search.best_params_)
print(f"Training Time : {lgbm_training_time:.3f} seconds")

joblib.dump(lgbm, "lgbm_model.joblib")
joblib.dump(lgbm_training_time, "lgbm_training_time.joblib")
print("Saved lgbm_model.joblib")

Training LightGBM...
Best params: {'subsample': 1.0, 'num_leaves': 31, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}
Training Time : 114.594 seconds
Saved lgbm_model.joblib


In [5]:
# ============================================================
# RANDOM FOREST MODEL
# ============================================================
rf_param_dist = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=15, cv=3, scoring="f1", random_state=42, n_jobs=-1
)

In [6]:
# ============================================================
# TRAIN MODEL
# ============================================================
print("="*60)
print("Training Random Forest...")
print("="*60)

start = time.time()
rf_search.fit(X_train_df, y_train)
end = time.time()

rf = rf_search.best_estimator_
rf_training_time = end - start

print("Best params:", rf_search.best_params_)
print(f"Training Time : {rf_training_time:.3f} seconds")

joblib.dump(rf, "rf_model.joblib")
joblib.dump(rf_training_time, "rf_training_time.joblib")
print("Saved rf_model.joblib")

Training Random Forest...
Best params: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}
Training Time : 651.381 seconds
Saved rf_model.joblib


In [7]:
# ============================================================
# MLP MODEL
# ============================================================
mlp_param_dist = {
    "hidden_layer_sizes": [(64,), (128, 64), (128, 64, 32)],
    "alpha": [1e-4, 1e-3, 1e-2],
    "learning_rate_init": [1e-3, 1e-2],
}

mlp_search = RandomizedSearchCV(
    MLPClassifier(max_iter=500, random_state=42),
    param_distributions=mlp_param_dist,
    n_iter=10, cv=3, scoring="f1", random_state=42, n_jobs=-1
)

In [8]:
# ============================================================
# TRAIN MODEL
# ============================================================
print("="*60)
print("Training MLP...")
print("="*60)

start = time.time()
mlp_search.fit(X_train_df, y_train)
end = time.time()

mlp = mlp_search.best_estimator_
mlp_training_time = end - start

print("Best params:", mlp_search.best_params_)
print(f"Training Time : {mlp_training_time:.3f} seconds")

joblib.dump(mlp, "mlp_model.joblib")
joblib.dump(mlp_training_time, "mlp_training_time.joblib")
print("Saved mlp_model.joblib")

Training MLP...
Best params: {'learning_rate_init': 0.01, 'hidden_layer_sizes': (128, 64, 32), 'alpha': 0.001}
Training Time : 958.570 seconds
Saved mlp_model.joblib
